# Emergency Referrals Analysis

This notebook analyzes entries from health_bench_data.jsonl that have the theme `emergency_referrals`.


In [17]:
import json
import pandas as pd
import re
from IPython.display import display, HTML, Markdown
from pprint import pprint


In [18]:
# Load the JSONL file
data = []
with open('health_bench_data.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Total entries loaded: {len(data)}")


Total entries loaded: 5000


In [19]:
# Filter for emergency_referrals theme
emergency_entries = []
for entry in data:
    if 'example_tags' in entry:
        tags = entry['example_tags']
        if any('emergency_referrals' in tag for tag in tags):
            emergency_entries.append(entry)

print(f"Found {len(emergency_entries)} entries with emergency_referrals theme")


Found 482 entries with emergency_referrals theme


## Emergency Referrals Examples

Below are formatted examples from the emergency_referrals dataset:


In [20]:
def display_entry(entry, index):
    """Display a single entry in a nicely formatted way"""
    display(Markdown(f"### Example {index + 1}"))
    display(Markdown("---"))
    
    # Display basic info
    display(Markdown(f"**Prompt ID:** `{entry.get('prompt_id', 'N/A')}`"))
    display(Markdown(f"**Tags:** {', '.join(entry.get('example_tags', []))}"))
    
    # Display the conversation prompt
    if 'prompt' in entry and entry['prompt']:
        display(Markdown("\n#### Conversation:"))
        for msg in entry['prompt']:
            role = msg.get('role', 'unknown').upper()
            content = msg.get('content', '')
            if role == 'USER':
                display(Markdown(f"**USER:** {content}"))
            elif role == 'ASSISTANT':
                display(Markdown(f"**ASSISTANT:** {content}"))
    
    # Display rubrics/criteria
    if 'rubrics' in entry and entry['rubrics']:
        display(Markdown("\n#### Evaluation Criteria:"))
        for i, rubric in enumerate(entry['rubrics'][:5], 1):  # Show first 5 criteria
            criterion = rubric.get('criterion', 'N/A')
            points = rubric.get('points', 0)
            tags = ', '.join(rubric.get('tags', []))
            
            # Truncate long criteria
            if len(criterion) > 200:
                criterion = criterion[:200] + "..."
            
            display(Markdown(f"**{i}. [{points} points]** {criterion}"))
            if tags:
                display(Markdown(f"   *Tags: {tags}*"))
        
        if len(entry['rubrics']) > 5:
            display(Markdown(f"\n   *... and {len(entry['rubrics']) - 5} more criteria*"))
    
    # Display ideal completion if available
    if 'ideal_completions_data' in entry and entry['ideal_completions_data']:
        ideal_data = entry['ideal_completions_data']
        if 'ideal_completion' in ideal_data and ideal_data['ideal_completion']:
            display(Markdown("\n#### Ideal Completion Sample:"))
            ideal_text = ideal_data['ideal_completion']
            # Show first 500 characters
            if len(ideal_text) > 500:
                ideal_text = ideal_text[:500] + "..."
            display(Markdown(f"```\n{ideal_text}\n```"))
    
    display(Markdown("\n" + "="*80 + "\n"))


## Pregnancy-Related Emergency Referrals

Filtering for conversations that mention pregnancy-related topics (using word boundary matching to avoid false positives):


In [23]:
# Filter for pregnancy-related entries based on user queries only
# Using word boundaries to avoid false positives (e.g., "elaborate" containing "labor")
pregnancy_keywords = [
    r'\bpregnan\w*\b',  # pregnant, pregnancy
    r'\bpostpartum\b',
    r'\bprenatal\b',
    r'\btrimester\b',
    r'\bgestation\w*\b',
    r'\bmaternal\b',
    r'\bfetal\b',
    r'\bbaby\b',
    r'\bbabies\b',
    r'\bbirth\b',
    r'\blabor\b',  # word boundary prevents matching "elaborate"
    r'\bdelivery\b',
    r'\bexpecting\b',
    r'\bmiscarriage\b',
    r'\babortion\b',
    r'\bectopic\b',
    r'\bpreeclampsia\b',
    r'\bnewborn\b',
    r'\binfant\b'
]

pregnancy_entries = []
for entry in emergency_entries:
    # Check only user messages in the prompt
    if 'prompt' in entry and entry['prompt']:
        user_messages = [msg.get('content', '').lower() for msg in entry['prompt'] if msg.get('role') == 'user']
        user_text = ' '.join(user_messages)
        
        # Use regex with word boundaries for precise matching
        if any(re.search(pattern, user_text) for pattern in pregnancy_keywords):
            pregnancy_entries.append(entry)

print(f"Found {len(pregnancy_entries)} pregnancy-related entries out of {len(emergency_entries)} emergency_referrals")


Found 33 pregnancy-related entries out of 482 emergency_referrals


In [24]:
# Display all pregnancy-related entries
display(Markdown(f"## All {len(pregnancy_entries)} Pregnancy-Related Emergency Referrals\n"))

for i, entry in enumerate(pregnancy_entries):
    display_entry(entry, i)


## All 33 Pregnancy-Related Emergency Referrals


### Example 1

---

**Prompt ID:** `be70739f-234a-4a0f-9b14-d15b8800577b`

**Tags:** theme:emergency_referrals, physician_agreed_category:non-emergent

...

   *Tags: level:example, axis:completeness*


   *... and 6 more criteria*


#### Ideal Completion Sample:

```
Difficulties breathing in a newborn are a serious medical condition and your baby should be examined by a professional in pediatrics or neonatology as soon as possible. Go to the closest ER or call 911 if you need assistance. 

What you describe can be a sign of apnea, milk aspiration or respiratory infection. Paleness around the lips suggests that your baby is not getting enough oxygen and that is why it is essential to correctly diagnose and treat your baby by a medical professional as soon as...
```


================================================================================


In [25]:
# Prepare data for CSV export
export_data = []

for entry in pregnancy_entries:
    # Extract physician category from tags
    physician_category = 'unknown'
    for tag in entry.get('example_tags', []):
        if 'physician_agreed_category:' in tag:
            physician_category = tag.split('physician_agreed_category:')[1]
            break
    
    # Get user messages only
    user_messages = []
    assistant_messages = []
    if 'prompt' in entry and entry['prompt']:
        for msg in entry['prompt']:
            if msg.get('role') == 'user':
                user_messages.append(msg.get('content', ''))
            elif msg.get('role') == 'assistant':
                assistant_messages.append(msg.get('content', ''))
    
    # Combine user messages
    full_user_query = '\n---\n'.join(user_messages)
    full_assistant_response = '\n---\n'.join(assistant_messages)
    
    # Get first criterion as example
    first_criterion = ''
    if 'rubrics' in entry and entry['rubrics'] and len(entry['rubrics']) > 0:
        first_criterion = entry['rubrics'][0].get('criterion', '')
    
    # Get ideal completion if available
    ideal_completion = ''
    if 'ideal_completions_data' in entry and entry['ideal_completions_data']:
        ideal_data = entry['ideal_completions_data']
        if 'ideal_completion' in ideal_data:
            ideal_completion = ideal_data['ideal_completion'] or ''
    
    export_data.append({
        'physician_agreed_category': physician_category,
        'prompt_id': entry.get('prompt_id', ''),
        'all_tags': ', '.join(entry.get('example_tags', [])),
        'user_query': full_user_query,
        'assistant_response': full_assistant_response,
        'num_criteria': len(entry.get('rubrics', [])),
        'example_criterion': first_criterion[:300],  # Truncate for readability
        'has_ideal_completion': bool(ideal_completion),
        'ideal_completion_preview': ideal_completion[:300] if ideal_completion else ''
    })

# Create DataFrame and sort by physician category
df = pd.DataFrame(export_data)
df = df.sort_values('physician_agreed_category')

# Save to CSV
output_filename = 'pregnancy_emergency_referrals.csv'
df.to_csv(output_filename, index=False)

print(f"Exported {len(df)} pregnancy-related entries to {output_filename}")
print(f"\nCategory breakdown:")
print(df['physician_agreed_category'].value_counts())


Exported 33 pregnancy-related entries to pregnancy_emergency_referrals.csv

Category breakdown:
physician_agreed_category
conditionally-emergent    13
emergent                  10
non-emergent               8
unknown                    2
Name: count, dtype: int64
